# **Caro PolicyNet — Google Colab Training**

Notebook huấn luyện PolicyNet theo contract `caro-v2`. Dataset được tải từ link Google Drive; checkpoint và lịch sử train được lưu vào Google Drive của người chạy.

# **0. Import libraries**

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

print({
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})

# **1. User configuration**

Chỉ cần sửa các giá trị trong cell dưới đây. `DATASET_DRIVE_URL` nhận link thư mục Google Drive được chia sẻ hoặc link file `.zip`. `DRIVE_OUTPUT_DIR` là nơi lưu trọng số trên My Drive.

In [ ]:
# ======================== CHỈNH CÁC GIÁ TRỊ Ở ĐÂY ========================
DATASET_DRIVE_URL = 'PASTE_GOOGLE_DRIVE_FOLDER_OR_ZIP_LINK_HERE'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/Caro_AI_Project/policy_runs/model-v2-mcts-strength8'

REPO_URL = 'https://github.com/minhhailtv6a1/Team_Project_TTNTNC_2025_2026.git'
REPO_BRANCH = 'hai'  # Branch phải chứa code Task 5–6 đã push lên GitHub.
PROJECT_SUBDIR = 'Caro_AI_Project'

RUN_ID = 'policy-v2-mcts-strength8-colab'
TRAINING_SEED = 20261109
BATCH_SIZE = 256
MAX_EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5
NUM_WORKERS = 2

RUN_SMOKE_FIRST = True
RERUN_SMOKE = False
RUN_PRODUCTION_TRAIN = True
RESUME_IF_LAST_CHECKPOINT_EXISTS = True
# ========================================================================

LOCAL_REPO_DIR = Path('/content/Team_Project_TTNTNC_2025_2026')
LOCAL_DATASET_DOWNLOAD_ROOT = Path('/content/caro_policy_dataset')
LOCAL_SMOKE_OUTPUT_DIR = Path('/content/caro_policy_smoke')
OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

assert DATASET_DRIVE_URL.startswith('http'), (
    'Hãy dán link Google Drive vào DATASET_DRIVE_URL trước khi chạy.'
)
print({'device': DEVICE, 'drive_output': str(OUTPUT_DIR), 'branch': REPO_BRANCH})

# **2. Mount Google Drive**

Colab sẽ yêu cầu quyền truy cập My Drive. Checkpoint `best.pt`, `last.pt`, history và TensorBoard được lưu trong `DRIVE_OUTPUT_DIR`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert OUTPUT_DIR.is_dir(), f'Không tạo được output directory: {OUTPUT_DIR}'
print(f'Artifacts sẽ được lưu tại: {OUTPUT_DIR}')

# **3. Install Colab dependencies**

Chỉ cài các package còn thiếu; giữ nguyên bản PyTorch có CUDA do Colab cung cấp.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'gdown>=5.2', 'tensorboard>=2.17'],
    check=True,
)
import gdown
print({'gdown': gdown.__version__, 'tensorboard_ready': True})

# **4. Load project source code**

Clone đúng branch chứa `PolicyNet`, `train_policy.py` và `NeuralAgent`. Nếu thư mục đã tồn tại, notebook tái sử dụng để tránh ghi đè thay đổi trong runtime.

In [ ]:
if not (LOCAL_REPO_DIR / '.git').is_dir():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(LOCAL_REPO_DIR)],
        check=True,
    )
else:
    print(f'Reuse repository đã có: {LOCAL_REPO_DIR}')

PROJECT_ROOT = LOCAL_REPO_DIR / PROJECT_SUBDIR
required_files = [
    PROJECT_ROOT / 'scripts/train_policy.py',
    PROJECT_ROOT / 'core/ml/architectures.py',
    PROJECT_ROOT / 'core/agents/neural_ai.py',
]
missing = [str(path) for path in required_files if not path.is_file() or path.stat().st_size == 0]
assert not missing, (
    f'Branch {REPO_BRANCH!r} chưa có đủ code Task 5–6: {missing}. '
    'Hãy push code lên branch rồi chọn Runtime > Restart session và chạy lại.'
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print({'project_root': str(PROJECT_ROOT), 'train_script': str(required_files[0])})

# **5. Load dataset from Google Drive**

Hỗ trợ hai dạng link:

- Link thư mục Drive chứa `dataset_manifest.json`, `train.npz`, `val.npz`, `test.npz`.
- Link file `.zip` chứa cùng các artifact trên.

Folder/file phải bật quyền **Anyone with the link — Viewer**. Notebook tự tìm thư mục chứa `dataset_manifest.json`.

In [ ]:
LOCAL_DATASET_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
manifest_candidates = list(LOCAL_DATASET_DOWNLOAD_ROOT.rglob('dataset_manifest.json'))

if not manifest_candidates:
    is_folder_link = '/folders/' in DATASET_DRIVE_URL or '/drive/folders/' in DATASET_DRIVE_URL
    if is_folder_link:
        gdown.download_folder(
            url=DATASET_DRIVE_URL,
            output=str(LOCAL_DATASET_DOWNLOAD_ROOT),
            quiet=False,
            use_cookies=False,
            remaining_ok=True,
        )
    else:
        archive_path = LOCAL_DATASET_DOWNLOAD_ROOT / 'dataset_from_drive.zip'
        downloaded = gdown.download(
            url=DATASET_DRIVE_URL,
            output=str(archive_path),
            quiet=False,
            fuzzy=True,
        )
        assert downloaded and archive_path.is_file(), 'Không tải được dataset từ link Drive.'
        assert zipfile.is_zipfile(archive_path), (
            'Link file phải trỏ tới một file .zip, hoặc hãy dùng link folder Drive.'
        )
        extract_dir = LOCAL_DATASET_DOWNLOAD_ROOT / 'extracted'
        extract_dir.mkdir(exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(extract_dir)

manifest_candidates = list(LOCAL_DATASET_DOWNLOAD_ROOT.rglob('dataset_manifest.json'))
assert len(manifest_candidates) == 1, (
    f'Cần đúng 1 dataset_manifest.json, tìm thấy {len(manifest_candidates)}: {manifest_candidates}'
)
DATASET_DIR = manifest_candidates[0].parent
print({'dataset_dir': str(DATASET_DIR), 'files': sorted(path.name for path in DATASET_DIR.iterdir())})

# **6. Validate dataset contract and checksums**

Kiểm tra contract, board size, ba split và SHA-256 trước khi cấp dữ liệu cho model.

In [ ]:
from core.engine.interfaces import CONTRACT_VERSION
from core.ml.data_pipeline import sha256_file

MANIFEST_PATH = DATASET_DIR / 'dataset_manifest.json'
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
MANIFEST_SHA256 = sha256_file(MANIFEST_PATH)

assert manifest.get('contract_version') == CONTRACT_VERSION
assert manifest.get('board_size') == 11
assert manifest.get('encoder', {}).get('channels') == ['current_player', 'opponent', 'legal']
for split in ('train', 'val', 'test'):
    artifact = manifest.get('artifacts', {}).get(split)
    assert isinstance(artifact, dict), f'Missing artifact: {split}'
    tensor_path = DATASET_DIR / artifact['tensor']
    assert tensor_path.is_file(), tensor_path
    assert sha256_file(tensor_path) == artifact['tensor_sha256'], f'Checksum fail: {split}'

print({
    'dataset_version': manifest['dataset_version'],
    'manifest_sha256': MANIFEST_SHA256,
    'samples': {name: manifest['artifacts'][name]['samples'] for name in ('train', 'val', 'test')},
})

# **7. Inspect one dataset batch**

Dùng public loader để kiểm tra shape, dtype và tính hợp lệ của target. Sau khi xem mẫu, mảng lớn được giải phóng khỏi RAM notebook.

In [ ]:
from core.ml.data_pipeline import load_dataset_split

preview_arrays = load_dataset_split(DATASET_DIR, 'val', verify_checksum=True)
preview_count = min(8, len(preview_arrays['target']))
PREVIEW_X = torch.from_numpy(preview_arrays['x'][:preview_count].copy())
PREVIEW_MASK = torch.from_numpy(preview_arrays['legal_mask'][:preview_count].copy())
PREVIEW_TARGET = torch.from_numpy(preview_arrays['target'][:preview_count].copy())

assert PREVIEW_MASK[torch.arange(preview_count), PREVIEW_TARGET].all()
print({
    'x': (tuple(PREVIEW_X.shape), str(PREVIEW_X.dtype)),
    'legal_mask': (tuple(PREVIEW_MASK.shape), str(PREVIEW_MASK.dtype)),
    'target': (tuple(PREVIEW_TARGET.shape), str(PREVIEW_TARGET.dtype)),
    'targets_are_legal': True,
})
del preview_arrays
gc.collect()

# **8. Build the training configuration**

Tạo config dựa trên manifest vừa tải, tránh hard-code checksum. Một bản config được lưu cùng checkpoint trên Drive để tái lập run.

In [ ]:
TRAIN_CONFIG = {
    'contract_version': CONTRACT_VERSION,
    'run_id': RUN_ID,
    'dataset': {
        'version': manifest['dataset_version'],
        'manifest_sha256': MANIFEST_SHA256,
    },
    'model': {
        'architecture_version': 'caro-policynet-resnet-v1',
        'board_size': 11,
        'input_channels': 3,
        'channels': 64,
        'residual_blocks': 4,
        'policy_channels': 32,
        'dropout': 0.1,
    },
    'training': {
        'seed': TRAINING_SEED,
        'batch_size': BATCH_SIZE,
        'epochs': MAX_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'num_workers': NUM_WORKERS,
    },
}

LOCAL_CONFIG_PATH = Path('/content/policy_train_colab.json')
config_text = json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2) + '\n'
LOCAL_CONFIG_PATH.write_text(config_text, encoding='utf-8')
(OUTPUT_DIR / 'train_config.json').write_text(config_text, encoding='utf-8')
print(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2))

# **9. Run a PolicyNet forward/backward check**

Kiểm tra model, legal mask, loss và gradient trên mini-batch trước khi chạy script huấn luyện.

In [ ]:
from core.ml.architectures import PolicyNet, PolicyNetConfig, masked_policy_loss

model_config = PolicyNetConfig.from_mapping(TRAIN_CONFIG['model'])
preview_model = PolicyNet(model_config).to(DEVICE)
preview_logits = preview_model(PREVIEW_X.to(DEVICE))
preview_loss = masked_policy_loss(
    preview_logits,
    PREVIEW_MASK.to(DEVICE),
    PREVIEW_TARGET.to(DEVICE),
)
preview_loss.backward()
gradients_finite = all(
    torch.isfinite(parameter.grad).all()
    for parameter in preview_model.parameters()
    if parameter.grad is not None
)
print({
    'logits_shape': tuple(preview_logits.shape),
    'loss': float(preview_loss.detach().cpu()),
    'gradients_finite': bool(gradients_finite),
    'parameter_count': sum(parameter.numel() for parameter in preview_model.parameters()),
})
del preview_model, preview_logits, preview_loss, PREVIEW_X, PREVIEW_MASK, PREVIEW_TARGET
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# **10. Run the smoke training gate**

Smoke run dùng output local riêng, không làm bẩn hoặc resume nhầm production checkpoint trên Drive. Tắt bằng `RUN_SMOKE_FIRST = False`.

In [ ]:
SMOKE_BEST_PATH = LOCAL_SMOKE_OUTPUT_DIR / 'best.pt'
if RUN_SMOKE_FIRST and (RERUN_SMOKE or not SMOKE_BEST_PATH.is_file()):
    smoke_command = [
        sys.executable, str(PROJECT_ROOT / 'scripts/train_policy.py'),
        '--config', str(LOCAL_CONFIG_PATH),
        '--dataset', str(DATASET_DIR),
        '--output', str(LOCAL_SMOKE_OUTPUT_DIR),
        '--device', DEVICE,
        '--smoke',
    ]
    print(' '.join(smoke_command))
    subprocess.run(smoke_command, cwd=PROJECT_ROOT, check=True)
elif RUN_SMOKE_FIRST:
    print(f'Reuse smoke checkpoint: {SMOKE_BEST_PATH}')
else:
    print('Smoke training đã được tắt trong cell cấu hình.')

if RUN_SMOKE_FIRST:
    smoke_summary = json.loads((LOCAL_SMOKE_OUTPUT_DIR / 'run_summary.json').read_text(encoding='utf-8'))
    print(json.dumps(smoke_summary, ensure_ascii=False, indent=2))

# **11. Train or resume the production model**

Nếu `last.pt` đã có trong `DRIVE_OUTPUT_DIR`, notebook tự resume khi `RESUME_IF_LAST_CHECKPOINT_EXISTS = True`. Test split chỉ được script đọc sau khi validation đã chọn `best.pt`.

In [ ]:
LAST_CHECKPOINT_PATH = OUTPUT_DIR / 'last.pt'
BEST_CHECKPOINT_PATH = OUTPUT_DIR / 'best.pt'

training_needed = RUN_PRODUCTION_TRAIN
if RUN_PRODUCTION_TRAIN and RESUME_IF_LAST_CHECKPOINT_EXISTS and LAST_CHECKPOINT_PATH.is_file():
    last_progress = torch.load(LAST_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    reached_epoch_limit = int(last_progress.get('epoch', 0)) >= MAX_EPOCHS
    reached_early_stop = int(last_progress.get('stale_epochs', 0)) >= EARLY_STOPPING_PATIENCE
    if reached_epoch_limit or reached_early_stop:
        training_needed = False
        print({
            'training_already_complete': True,
            'last_epoch': last_progress.get('epoch'),
            'best_epoch': last_progress.get('best_epoch'),
            'early_stopped': reached_early_stop,
        })

if training_needed:
    train_command = [
        sys.executable, str(PROJECT_ROOT / 'scripts/train_policy.py'),
        '--config', str(LOCAL_CONFIG_PATH),
        '--dataset', str(DATASET_DIR),
        '--output', str(OUTPUT_DIR),
        '--device', DEVICE,
    ]
    if RESUME_IF_LAST_CHECKPOINT_EXISTS and LAST_CHECKPOINT_PATH.is_file():
        train_command.extend(['--resume', str(LAST_CHECKPOINT_PATH)])
        print(f'Resume từ: {LAST_CHECKPOINT_PATH}')
    else:
        print('Bắt đầu production run mới.')
    print(' '.join(train_command))
    subprocess.run(train_command, cwd=PROJECT_ROOT, check=True)
elif not RUN_PRODUCTION_TRAIN:
    print('Production training đã được tắt trong cell cấu hình.')

# **12. Review training metrics and learning curves**

Đọc kết quả trực tiếp từ Drive. Nếu Colab mất kết nối, chạy lại các cell đầu và cell này để xem run đã lưu.

In [ ]:
SUMMARY_PATH = OUTPUT_DIR / 'run_summary.json'
HISTORY_PATH = OUTPUT_DIR / 'history.csv'

if SUMMARY_PATH.is_file():
    run_summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
    print(json.dumps(run_summary, ensure_ascii=False, indent=2))
else:
    print(f'Chưa có run summary tại {SUMMARY_PATH}')

if HISTORY_PATH.is_file():
    history = pd.read_csv(HISTORY_PATH)
    display(history.tail(10))
    axes = history.plot(
        x='epoch',
        y=['train_loss', 'val_loss'],
        marker='o',
        figsize=(10, 5),
        title='PolicyNet learning curves',
    )
    axes.grid(True, alpha=0.3)
    plt.show()

# **13. Open TensorBoard**

TensorBoard đọc event files đã lưu trong output Drive.

In [ ]:
from tensorboard import notebook as tensorboard_notebook

TENSORBOARD_DIR = OUTPUT_DIR / 'tensorboard'
if TENSORBOARD_DIR.is_dir():
    tensorboard_notebook.start(f'--logdir {TENSORBOARD_DIR}')
else:
    print(f'Chưa có TensorBoard directory: {TENSORBOARD_DIR}')

# **14. Validate the best checkpoint with NeuralAgent**

Load `best.pt`, đối chiếu dataset checksum và kiểm tra agent trả một opening action hợp lệ mà không mutation environment.

In [ ]:
from core.agents.neural_ai import NeuralAgent
from core.engine.game_env import CaroEnv

assert BEST_CHECKPOINT_PATH.is_file(), f'Không tìm thấy {BEST_CHECKPOINT_PATH}'
checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
assert checkpoint['dataset_version'] == manifest['dataset_version']
assert checkpoint['dataset_manifest_sha256'] == MANIFEST_SHA256

agent = NeuralAgent(
    BEST_CHECKPOINT_PATH,
    device=DEVICE,
    expected_dataset_version=manifest['dataset_version'],
    expected_dataset_manifest_sha256=MANIFEST_SHA256,
)
env = CaroEnv()
before = env.state
decision = agent.choose_action(env)
print({
    'action': decision.action,
    'score': decision.score,
    'legal': decision.action in env.legal_actions(),
    'no_mutation': env.state == before,
    'best_epoch': checkpoint.get('best_epoch'),
    'test_metrics': checkpoint.get('test_metrics'),
})

# **15. List saved Drive artifacts**

Xác nhận các artifact cần giữ sau khi runtime Colab kết thúc.

In [ ]:
saved_artifacts = []
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        saved_artifacts.append({
            'path': str(path.relative_to(OUTPUT_DIR)),
            'size_mb': round(path.stat().st_size / (1024 ** 2), 3),
        })
display(pd.DataFrame(saved_artifacts))
print(f'Hoàn tất. Toàn bộ artifact nằm tại: {OUTPUT_DIR}')